# Assignment 4: QUBO – Select 10 Assets from 30 Stocks
This notebook formulates the portfolio selection problem as a Quadratic Unconstrained Binary Optimization (QUBO) problem. 

**Objective:** Maximize return by selecting exactly 10 assets out of a 30-stock universe.
**Data:** Historical daily closing prices. We will dynamically select a time window to calculate the expected historical mean returns ($r_i$).

We convert the cardinality constraint ($\sum_{i=1}^{N} x_i = 10$) into a penalty term to form the final QUBO objective:
$$\min \left( -\sum_{i=1}^{N} r_i x_i + \lambda \left( \sum_{i=1}^{N} x_i - 10 \right)^2 \right)$$

In [13]:
# Install the dimod library required for Simulated Annealing
!pip install dimod

## 1. Load Data & Calculate Historical Mean Returns
We will load the CSV, set the date column as our index, and convert it to proper datetime objects. Then, we use the `years_to_use` variable to slice the data. 

*Note: We use `ffill()` to forward-fill missing prices before calculating daily percentage changes. This prevents missing data (like early years of KOTAKBANK or LT) from breaking the return calculations.*

In [14]:
import pandas as pd
import numpy as np
import dimod

### --- PARAMETER ---
#### Choose how many years of recent data you want to use for the mean return calculation

In [15]:
years_to_use = 5  # Change this to 1, 5, 10, 16, etc.

### Load the dataset. index_col=0 automatically sets 'Unnamed: 0' as the index.

In [33]:
df = pd.read_csv("NSE_stock_data.csv", index_col=0, parse_dates=True)

### Ensure the index is in datetime format

In [17]:
df.index = pd.to_datetime(df.index)

### Find the latest date in the dataset and calculate the start date

In [18]:
latest_date = df.index.max()
start_date = latest_date - pd.DateOffset(years=years_to_use)

In [19]:
# Filter the dataframe for the selected time window
df_filtered = df[df.index >= start_date].copy()

### Calculate daily returns (forward-fill missing prices first)

In [20]:
daily_returns = df_filtered.ffill().pct_change().dropna()

### Calculate historical mean returns (Annualized assuming 252 trading days)

In [21]:
mean_returns = daily_returns.mean() * 252
r = mean_returns.values
tickers = mean_returns.index.tolist()

In [22]:
N = len(tickers)
print(f"Total Universe Size (N): {N}")
print(f"Data timeframe: {start_date.strftime('%Y-%m-%d')} to {latest_date.strftime('%Y-%m-%d')} ({years_to_use} years)")
print("\nSample Expected Returns (r_i):")
print(mean_returns.head())

Total Universe Size (N): 30
Data timeframe: 2016-04-30 to 2021-04-30 (5 years)

Sample Expected Returns (r_i):
ASIANPAINT    0.171595
BPCL         -0.026333
BRITANNIA     0.074481
CIPLA         0.099737
DRREDDY       0.099564
dtype: float64


## 2. Build QUBO Matrix $Q$
We need to build the QUBO matrix $Q$ for the full universe of 30 stocks.
The penalty parameter $\lambda$ must be large enough to enforce feasibility. We will set $\lambda$ to 15× the average magnitude of the returns.

Based on expanding $(\sum x_i - 10)^2$, the QUBO weights are:
* **Diagonal Terms ($Q_{ii}$):** $-r_i + \lambda(1 - 2(10)) = -r_i - 19\lambda$
* **Off-Diagonal Terms ($Q_{ij}$):** $2\lambda$ for $i \neq j$

In [23]:
k = 10 # Target portfolio size

# Calculate penalty parameter lambda
# Using 15x the average absolute return to ensure the penalty is respected
lam = 15 * np.mean(np.abs(r))
print(f"Penalty Parameter (Lambda): {lam:.4f}")

# Initialize QUBO dictionary (dimod expects a dict of {(i, j): weight})
Q = {}

# Populate the QUBO matrix
for i in range(N):
    # Diagonal Terms: -r_i - 19*lambda
    Q[(i, i)] = -r[i] - 19 * lam 
    
    # Off-Diagonal Terms: 2*lambda
    for j in range(i + 1, N):
        Q[(i, j)] = 2 * lam

Penalty Parameter (Lambda): 1.2991


## 3. Solve using Simulated Annealing
Because checking all combinations for 30 stocks ($2^{30} \approx 1.07 \text{ billion}$) is too computationally heavy for brute force, we use `dimod`'s `SimulatedAnnealingSampler`. This heuristic algorithm navigates the quadratic landscape to quickly find the global minimum.

In [24]:
# Initialize the simulated annealing solver
sampler = dimod.SimulatedAnnealingSampler()

# Solve the QUBO (num_reads determines how many times the annealing is run)
sampleset = sampler.sample_qubo(Q, num_reads=100)

# Extract the best sample (the one with the lowest energy/minimum value)
best_sample = sampleset.first.sample
best_energy = sampleset.first.energy

# Identify selected assets where x_i = 1
selected_indices = [i for i, val in best_sample.items() if val == 1]
selected_stocks = [tickers[i] for i in selected_indices]

print(f"Number of assets selected: {len(selected_stocks)}")
print(f"Selected Assets: {selected_stocks}")

Number of assets selected: 10
Selected Assets: ['ASIANPAINT', 'HDFC', 'ICICIBANK', 'ITC', 'KOTAKBANK', 'LT', 'RELIANCE', 'SHREECEM', 'TATAMOTORS', 'TITAN']


## 4. Verify and Compare
Finally, we verify that exactly 10 assets were selected and compare our QUBO selected assets with the top 10 assets ranked purely by their expected return $r_i$.

In [25]:
# 1. Verify constraint
assert len(selected_stocks) == 10, f"Constraint failed! Selected {len(selected_stocks)} assets instead of 10."
print("✅ Verification Passed: Exactly 10 assets selected.\n")

# 2. Compare with Top 10 by Return
# Get indices of the top 10 expected returns
top_10_indices = np.argsort(r)[-10:][::-1]
top_10_stocks = [tickers[i] for i in top_10_indices]

# Display comparison
comparison_df = pd.DataFrame({
    'QUBO Selected (10)': selected_stocks,
    'QUBO Expected Return': [r[i] for i in selected_indices],
    'Top 10 Ranked by r_i': top_10_stocks,
    'Top 10 Expected Return': [r[i] for i in top_10_indices]
})

print(comparison_df.to_string(index=False))

# Check the overlap
overlap = set(selected_stocks).intersection(set(top_10_stocks))
print(f"\nOverlap between QUBO and Top 10 ranked: {len(overlap)} stocks.")

✅ Verification Passed: Exactly 10 assets selected.

QUBO Selected (10)  QUBO Expected Return Top 10 Ranked by r_i  Top 10 Expected Return
        ASIANPAINT              0.171595                TITAN                0.235725
              HDFC              0.142022            ICICIBANK                0.174343
         ICICIBANK              0.174343           ASIANPAINT                0.171595
               ITC             -0.031380             RELIANCE                0.156869
         KOTAKBANK              0.151010            KOTAKBANK                0.151010
                LT              0.046101                 HDFC                0.142022
          RELIANCE              0.156869             SHREECEM                0.140087
          SHREECEM              0.140087                 SBIN                0.134672
        TATAMOTORS              0.029871           ULTRACEMCO                0.123990
             TITAN              0.235725               MARUTI                0.105850

O



### **1. What happens if $\lambda$ is too small?** 

Think of $\lambda$ as a "fine" for breaking the rules. If the penalty parameter $\lambda$ is too small, the penalty is too weak. The optimizer will prioritize maximizing returns (the linear terms) over satisfying the constraint. As a result, it might select more than 10 assets because the "reward" of capturing more positive returns mathematically outweighs the tiny penalty for breaking the rule.



### **2. What happens if $\lambda$ is too large?** 

If $\lambda$ is too large, the penalty completely dominates the objective function. The optimizer will become hyper-focused on bringing that massive penalty term to exactly zero to avoid the fine. It will successfully pick exactly 10 stocks, but it will practically ignore the expected returns ($r_i$) because they look like tiny rounding errors compared to the massive penalty. You will get 10 stocks, but it might be a mathematically suboptimal combination regarding returns.



### **3. Why does this formulation produce quadratic interactions?** 

This happens purely because of the algebra required to turn the constraint into a penalty. [cite_start]To enforce the rule (pick exactly 10), we convert the constraint into a penalty term by squaring it: $\lambda (\sum_{i=1}^{N} x_i - 10)^2$[cite: 21]. [cite_start]When you expand a squared sum mathematically, it naturally generates cross-terms ($x_i x_j$)[cite: 28]. [cite_start]In the QUBO model, these quadratic interaction terms are what actually enforce the cardinality[cite: 42]. They act as a communication network, essentially saying: *"If stock $i$ is selected, adjust the probability of selecting stock $j$ so we don't accidentally pick 11 stocks."*

### **4. How would you add risk (covariance matrix $\Sigma$) to this QUBO?** 
Adding risk fits perfectly into a QUBO because risk is inherently quadratic! In portfolio optimization, risk is measured by how stocks move together, captured by the covariance matrix ($\Sigma$), with portfolio variance calculated as $\sum_i \sum_j \Sigma_{ij} x_i x_j$. To add this, you would introduce a risk aversion parameter (like $\gamma$) and simply add $+ \gamma \Sigma_{ij}$ directly into your existing off-diagonal terms ($Q_{ij}$) of the QUBO matrix.